# 6. Agents and Tools

Let the model choose between lookup and calculation tools.

[🔊 Open Audio Practice](https://htmlpreview.github.io/?https://github.com/blaire101/langchain-course-companion-26/blob/main/docs/06_agents_audio.html)

> GitHub does not reliably execute Mermaid or custom JavaScript inside notebook previews.  
> This notebook therefore uses a static PNG diagram, while pronunciation practice is provided in a companion HTML page.


In [ ]:
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain.tools import tool
from dotenv import load_dotenv

load_dotenv()
MODEL = "openai:gpt-4o-mini"


## 1. Define the tools

A tool needs:

1. A clear **name**
2. Accurate **type hints**
3. A precise **docstring**

These elements form the tool schema used by the model during tool selection.


In [ ]:
@tool
def plan_lookup(plan: str) -> str:
    """Look up known features and monthly price for a subscription plan."""
    data = {
        "starter": {"price": 29, "features": "five users"},
        "business": {"price": 99, "features": "audit logs, API access"},
    }
    return str(data.get(plan.lower(), "Unknown plan"))


@tool
def total_cost(monthly_price: float, months: int) -> float:
    """Calculate total subscription cost."""
    return monthly_price * months


## 2. Create the agent

The agent receives a model, a list of tools, and a system instruction.  
Unlike a fixed chain, it decides dynamically which tool to call and in what order.


In [ ]:
agent = create_agent(
    model=init_chat_model(MODEL),
    tools=[plan_lookup, total_cost],
    system_prompt="Use tools for plan facts and calculations.",
)


## 3. Invoke the agent

In [ ]:
question = (
    "What does the Business plan include "
    "and what is the cost for 12 months?"
)

result = agent.invoke(
    {"messages": [{"role": "user", "content": question}]}
)

print(result["messages"][-1].content)


### Example output

```text
The Business plan includes audit logs and API access.
At $99 per month, the total cost for 12 months is $1,188.
```

The wording may vary, but the facts and calculation should remain consistent.


## 4. Inspect the tool-calling trace

The returned message list can contain the user request, AI tool calls, tool results, and the final AI response.


In [ ]:
for index, message in enumerate(result["messages"], start=1):
    print(f"\n--- Message {index}: {type(message).__name__} ---")
    print(message)


### Typical execution trace

```text
User
  → asks for Business plan features and 12-month cost

Agent
  → calls plan_lookup(plan="Business")

plan_lookup
  → returns price=99 and features="audit logs, API access"

Agent
  → calls total_cost(monthly_price=99, months=12)

total_cost
  → returns 1188

Agent
  → writes the final customer-facing answer
```


## 5. Relationship Diagram

![Agent and tools relationship](../assets/agent_tool_flow.png)

### Core relationship

- **Agent** = decision maker
- **Tool** = executable capability
- **Tool result** = trusted observation
- **Final answer** = response generated from tool results


## 6. Think Summary

### How do tool names, type hints, and docstrings affect tool selection?

**Tool name**  
A descriptive name helps the model infer purpose.

**Type hints**  
Type hints define the expected argument schema.

**Docstring**  
The docstring explains when the tool should be used.

### Key takeaway

A strong tool has a clear purpose, narrow responsibility, predictable inputs, and a reliable output.


## 7. Interview Q&A

### Q1. What is the difference between a chain and an agent?

A chain follows a predefined sequence. An agent dynamically chooses tools and the next action.

### Q2. Does the LLM execute the Python function itself?

No. The model selects the tool and generates arguments. The application runtime executes the function.

### Q3. Why use a calculation tool instead of asking the LLM directly?

A calculation tool is deterministic, testable, and less error-prone.


## 8. How to Explain What You Built

### 30-second interview answer

I created a LangChain agent with two tools. The first retrieves subscription-plan facts, and the second calculates total cost. The agent selects the lookup tool first, passes the returned monthly price into the calculation tool, and then generates the final customer-facing answer.

### One-line résumé description

Built a LangChain agent that dynamically routed subscription queries across lookup and calculation tools.
